# Tool use optimization tutorial: Reducing API turns

This notebook demonstrates how different design decisions impact the number of API turns (requests to Claude) needed to complete tasks.

You'll see four progressive optimizations:
1. **Baseline**: Separate tools for stock price and calculation
2. **Optimization 1**: Combined portfolio tool
3. **Optimization 2**: Better system prompt for parallel execution
4. **Optimization 3**: Expression evaluator for complex math

Each optimization will show turn counts for the same queries so you can see the impact.

## Setup

In [ ]:
# Install required packages (uncomment if needed)
# !pip install anthropic python-dotenv

import ast
import json
import math
import operator
import os
from typing import Any, Dict, List, Optional
from getpass import getpass

from anthropic import Anthropic
from dotenv import load_dotenv

In [ ]:
# Load API key
load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    print("ANTHROPIC_API_KEY not found in environment.")
    print("Get your API key from: https://console.anthropic.com/")
    api_key = getpass("Enter your Anthropic API key: ")

if not api_key.startswith("sk-ant-"):
    raise ValueError("Invalid API key format. API key should start with 'sk-ant-'.")

client = Anthropic(api_key=api_key)
print("✓ API key configured successfully!")

In [ ]:
# Configuration
CLAUDE_MODEL = "claude-3-5-haiku-20241022"
MAX_TOKENS = 1024
TEMPERATURE = 0.0

# Stock price data
STOCK_PRICES: Dict[str, Dict[str, Any]] = {
    "aapl": {"price": 195.50, "name": "Apple"},
    "msft": {"price": 425.30, "name": "Microsoft"},
    "nvda": {"price": 875.20, "name": "NVIDIA"},
    "goog": {"price": 162.75, "name": "Alphabet (Google)"},
    "googl": {"price": 162.75, "name": "Alphabet (Google)"},
    "amzn": {"price": 185.40, "name": "Amazon"},
    "meta": {"price": 520.80, "name": "Meta (Facebook)"},
    "tsla": {"price": 245.60, "name": "Tesla"},
}

print(f"Model: {CLAUDE_MODEL}")
print(f"Temperature: {TEMPERATURE}")

## Version 1: Baseline (separate tools)

This version has separate tools for getting stock prices and doing calculations.

**Problem**: For portfolio questions, Claude needs to call `get_stock_price()` multiple times (once per stock), increasing the number of turns.

In [ ]:
# Version 1: Tool Functions

def get_stock_price_v1(ticker: str) -> Dict[str, Any]:
    """Get the price of a single stock."""
    ticker = ticker.lower()
    if ticker not in STOCK_PRICES:
        raise KeyError(f"Ticker '{ticker}' not found")
    
    return {
        "ticker": ticker,
        "name": STOCK_PRICES[ticker]["name"],
        "price": STOCK_PRICES[ticker]["price"],
    }

def calculate_v1(op: str, a: float, b: float) -> Dict[str, Any]:
    """Perform binary math operation."""
    match op:
        case "+":
            result = a + b
        case "-":
            result = a - b
        case "*":
            result = a * b
        case "/":
            if b == 0:
                raise ZeroDivisionError("Division by zero")
            result = a / b
        case "**":
            result = a**b
        case "log":
            if b <= 0 or b == 1 or a <= 0:
                raise ValueError(f"Invalid logarithm: log({a}, {b})")
            result = math.log(a, b)
        case _:
            raise ValueError(f"Unsupported operation: {op}")
    
    return {"operation": op, "a": a, "b": b, "result": result}

print("✓ Version 1 tools defined")

In [ ]:
# Version 1: Tool Specifications

get_stock_price_spec_v1 = {
    "name": "get_stock_price",
    "description": "Get the current price of a single stock by ticker symbol.",
    "input_schema": {
        "type": "object",
        "properties": {
            "ticker": {
                "type": "string",
                "description": "Stock ticker symbol (e.g., 'tsla', 'goog')",
            }
        },
        "required": ["ticker"],
    },
}

calculate_spec_v1 = {
    "name": "calculate",
    "description": "Perform precise mathematical calculations on two numbers.",
    "input_schema": {
        "type": "object",
        "properties": {
            "op": {
                "type": "string",
                "description": "Operation: '+', '-', '*', '/', '**', 'log'",
            },
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["op", "a", "b"],
    },
}

print("✓ Version 1 tool specs defined")

In [ ]:
# Version 1: System Prompt (basic)

SYSTEM_PROMPT_V1 = """You are a helpful finance assistant.

Use the available tools:
- get_stock_price: Look up stock prices
- calculate: Perform math calculations

After using tools, provide ONLY the final answer value with no additional text."""

print("✓ Version 1 system prompt defined")

In [ ]:
# Agent function with turn counting

def run_agent(prompt: str, system_prompt: str, tools_specs: list, tool_functions: dict, verbose: bool = True) -> tuple:
    """Run agent and return (result, turn_count)."""
    messages = [{"role": "user", "content": prompt}]
    turn = 0
    
    while True:
        turn += 1
        if verbose:
            print(f"  Turn {turn}", end="")
        
        response = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            system=system_prompt,
            tools=tools_specs,
            messages=messages,
        )
        
        messages.append({"role": "assistant", "content": response.content})
        
        tool_calls = [b for b in response.content if getattr(b, "type", None) == "tool_use"]
        
        if not tool_calls:
            # Final answer
            final_answer = "".join([
                getattr(b, "text", "")
                for b in response.content
                if getattr(b, "type", None) == "text"
            ])
            if verbose:
                print(" → Final answer")
            return final_answer.strip(), turn
        
        if verbose:
            print(f" → {len(tool_calls)} tool call(s): {[getattr(tc, 'name', '?') for tc in tool_calls]}")
        
        # Execute tools
        tool_results = []
        for tc in tool_calls:
            tool_name = getattr(tc, "name", None)
            tool_input = getattr(tc, "input", {})
            
            try:
                if tool_name in tool_functions:
                    result = tool_functions[tool_name](**tool_input)
                    content = json.dumps(result)
                else:
                    content = json.dumps({"error": f"Unknown tool: {tool_name}"})
            except Exception as e:
                content = json.dumps({"error": str(e)})
            
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tc.id,
                "content": content,
            })
        
        messages.append({"role": "user", "content": tool_results})

print("✓ Agent function defined")

### Test version 1: Portfolio query

Watch how many turns it takes to calculate a 3-stock portfolio.

In [ ]:
query = "How much would it cost to buy 15 shares of tesla, 24 shares of google, and 120 shares of amazon?"

print(f"Query: {query}\n")

tool_functions_v1 = {
    "get_stock_price": get_stock_price_v1,
    "calculate": calculate_v1,
}

result, turns = run_agent(
    query,
    SYSTEM_PROMPT_V1,
    [get_stock_price_spec_v1, calculate_spec_v1],
    tool_functions_v1
)

print(f"\n📊 Result: ${result}")
print(f"🔄 Total turns: {turns}")

## Version 2: Combined portfolio tool

**Optimization**: Replace `get_stock_price()` with `calculate_portfolio_value()` that can handle multiple stocks at once.

**Expected improvement**: Reduce 3+ API calls to 1 for portfolio queries.

In [ ]:
# Version 2: Combined Portfolio Tool

def calculate_portfolio_value_v2(stocks: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Calculate value of multiple stocks in one call."""
    breakdown = []
    total_value = 0.0

    for stock in stocks:
        ticker = stock["ticker"].lower()
        quantity = stock["quantity"]

        if ticker not in STOCK_PRICES:
            raise KeyError(f"Ticker '{ticker}' not found")

        price = STOCK_PRICES[ticker]["price"]
        value = price * quantity

        breakdown.append({
            "ticker": ticker,
            "name": STOCK_PRICES[ticker]["name"],
            "price": price,
            "quantity": quantity,
            "value": value,
        })
        total_value += value

    return {"breakdown": breakdown, "total_value": total_value}

print("✓ Version 2 tools defined")

In [ ]:
# Version 2: Tool Specification

_ticker_names = ", ".join([f"{t} ({d['name']})" for t, d in STOCK_PRICES.items()])

calculate_portfolio_value_spec_v2 = {
    "name": "calculate_portfolio_value",
    "description": (
        f"Calculate portfolio value for one or more stocks. "
        f"For a single stock, use quantity=1. "
        f"Available tickers: {_ticker_names}."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "stocks": {
                "type": "array",
                "description": "Array of stocks with 'ticker' and 'quantity' fields.",
                "items": {
                    "type": "object",
                    "properties": {
                        "ticker": {"type": "string", "description": "Stock ticker (e.g., 'tsla')"},
                        "quantity": {"type": "number", "description": "Number of shares"},
                    },
                    "required": ["ticker", "quantity"],
                },
            }
        },
        "required": ["stocks"],
    },
}

print("✓ Version 2 tool spec defined")

In [ ]:
# Version 2: Updated System Prompt

SYSTEM_PROMPT_V2 = """You are a helpful finance assistant.

Use the available tools:
- calculate_portfolio_value: Look up stock prices and calculate portfolio value. 
  For single stock price, use quantity=1. For multiple stocks, include ALL in one call.
- calculate: Perform math calculations

After using tools, provide ONLY the final answer value with no additional text."""

print("✓ Version 2 system prompt defined")

### Test version 2: Same portfolio query

In [ ]:
print(f"Query: {query}\n")

tool_functions_v2 = {
    "calculate_portfolio_value": calculate_portfolio_value_v2,
    "calculate": calculate_v1,
}

result, turns = run_agent(
    query,
    SYSTEM_PROMPT_V2,
    [calculate_portfolio_value_spec_v2, calculate_spec_v1],
    tool_functions_v2
)

print(f"\n📊 Result: ${result}")
print(f"🔄 Total turns: {turns}")
print(f"\n✅ Improvement: Reduced from ~5-6 turns to {turns} turns!")

## Version 3: Optimized system prompt for parallel execution

**Optimization**: Improve the system prompt to encourage Claude to make parallel tool calls when operations are independent.

**Example**: For "173 * 3232 + 342 / 72.1", Claude can call multiplication and division in parallel, then add the results.

**Expected improvement**: Reduce turns for complex math from 5-6 to 2-3.

In [ ]:
# Version 3: Optimized System Prompt

SYSTEM_PROMPT_V3 = """You are a helpful finance assistant.

Use the available tools:
- calculate_portfolio_value: Look up stock prices and calculate portfolio value. 
  For single stock price, use quantity=1. For multiple stocks, include ALL in one call.
- calculate: Perform math calculations

EFFICIENCY TIP - Parallel Tool Calls:
You can make multiple tool calls in a single response. When you identify operations 
that are independent (don't depend on each other's results), call them in parallel 
to reduce the number of turns.

Example: For "173 * 3232 + 342 / 72.1", the multiplication and division are independent,
so call both in parallel, then add the results.

After using tools, provide ONLY the final answer value with no additional text."""

print("✓ Version 3 system prompt defined")

### Test version 3: Complex math query

In [ ]:
math_query = "Help me solve this math problem: 173 * 3232 + 342 / 72.1"

print(f"Query: {math_query}\n")
print("Version 2 (without parallel hint):")
result_v2, turns_v2 = run_agent(
    math_query,
    SYSTEM_PROMPT_V2,
    [calculate_portfolio_value_spec_v2, calculate_spec_v1],
    tool_functions_v2
)
print(f"  Result: {result_v2}, Turns: {turns_v2}\n")

print("Version 3 (with parallel hint):")
result_v3, turns_v3 = run_agent(
    math_query,
    SYSTEM_PROMPT_V3,
    [calculate_portfolio_value_spec_v2, calculate_spec_v1],
    tool_functions_v2
)
print(f"  Result: {result_v3}, Turns: {turns_v3}")

if turns_v3 < turns_v2:
    print(f"\n✅ Improvement: Reduced from {turns_v2} to {turns_v3} turns!")
else:
    print(f"\n⚠️  No improvement (both {turns_v2} turns). Claude may have already optimized.")
    
print("\n💡 But can we do even better? See Version 4...")

## Version 4: Expression evaluator

**Optimization**: Add a safe expression evaluator tool that can handle entire math expressions in one call.

**Key insight**: Instead of breaking "173 * 3232 + 342 / 72.1" into multiple binary operations, evaluate the entire expression at once.

**Expected improvement**: Reduce complex math from 3 turns to 2 turns (call expression evaluator → return result).

**Tradeoff**: More complex tool implementation, but eliminates multiple calculation rounds.

In [ ]:
# Version 4: Safe Expression Evaluator using AST

def evaluate_math_expression_v4(expression: str) -> Dict[str, Any]:
    """
    Safely evaluate a mathematical expression using Python's AST module.
    
    This avoids security risks from eval() by explicitly whitelisting allowed operations.
    
    Supports:
    - Basic operations: +, -, *, /, ** (power)
    - Functions: sqrt(x), ln(x), log(x)
    - Parentheses for grouping
    
    Args:
        expression: Math expression as string (e.g., "173 * 3232 + 342 / 72.1")
    
    Returns:
        Dict with the expression and result
    """
    # Define safe binary operations
    safe_ops = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.USub: operator.neg,  # Unary minus
    }
    
    # Define safe functions
    safe_functions = {
        'sqrt': lambda x: x ** 0.5,
        'ln': math.log,
        'log': math.log10,
    }
    
    def eval_node(node):
        """Recursively evaluate AST nodes."""
        if isinstance(node, ast.Constant):  # Numbers
            return node.value
        elif isinstance(node, ast.BinOp):  # Binary operations like a + b
            left = eval_node(node.left)
            right = eval_node(node.right)
            return safe_ops[type(node.op)](left, right)
        elif isinstance(node, ast.UnaryOp):  # Unary operations like -x
            operand = eval_node(node.operand)
            return safe_ops[type(node.op)](operand)
        elif isinstance(node, ast.Call):  # Function calls like sqrt(x)
            func_name = node.func.id
            if func_name not in safe_functions:
                raise ValueError(f"Function '{func_name}' not allowed")
            args = [eval_node(arg) for arg in node.args]
            return safe_functions[func_name](*args)
        else:
            raise ValueError(f"Unsupported operation: {type(node).__name__}")
    
    try:
        # Parse expression into AST
        tree = ast.parse(expression, mode='eval')
        result = eval_node(tree.body)
        return {"expression": expression, "result": result}
    except Exception as e:
        raise ValueError(f"Failed to evaluate expression '{expression}': {str(e)}")

print("✓ Version 4 expression evaluator defined")

In [ ]:
# Version 4: Tool Specification

evaluate_math_expression_spec_v4 = {
    "name": "evaluate_math_expression",
    "description": (
        "Evaluate a complete mathematical expression in one call. "
        "Supports +, -, *, /, ** (power), parentheses, and functions: sqrt(), ln(), log(). "
        "Use this for complex expressions instead of breaking them into multiple calculate() calls. "
        "Examples: '173 * 3232 + 342 / 72.1' or 'sqrt(234.13) + ln(27389140.25) + 173 * 32 + 4.5**2'"
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "Mathematical expression to evaluate (e.g., '173 * 3232 + 342 / 72.1')",
            }
        },
        "required": ["expression"],
    },
}

print("✓ Version 4 tool spec defined")

In [ ]:
# Version 4: Updated System Prompt

SYSTEM_PROMPT_V4 = """You are a helpful finance assistant.

Use the available tools:
- calculate_portfolio_value: Look up stock prices and calculate portfolio value. 
  For single stock price, use quantity=1. For multiple stocks, include ALL in one call.
- evaluate_math_expression: Evaluate complete math expressions in one call.
  Use this for ANY math problem - it handles complex expressions with multiple operations.

After using tools, provide ONLY the final answer value with no additional text."""

print("✓ Version 4 system prompt defined")

### Test version 4: Complex math query

In [ ]:
print(f"Query: {math_query}\n")

tool_functions_v4 = {
    "calculate_portfolio_value": calculate_portfolio_value_v2,
    "evaluate_math_expression": evaluate_math_expression_v4,
}

print("Version 4 (expression evaluator):")
result_v4, turns_v4 = run_agent(
    math_query,
    SYSTEM_PROMPT_V4,
    [calculate_portfolio_value_spec_v2, evaluate_math_expression_spec_v4],
    tool_functions_v4
)
print(f"  Result: {result_v4}, Turns: {turns_v4}")

print(f"\n✅ Improvement: Reduced from {turns_v3} turns (V3) to {turns_v4} turns (V4)!")
print(f"   V1 (baseline): {turns_v1} turns")
print(f"   V3 (parallel): {turns_v3} turns") 
print(f"   V4 (expression): {turns_v4} turns")

### Test version 4: Very complex math query

In [ ]:
complex_math = "Help me solve: sqrt(234.13) + ln(27389140.25) + 173 * 32 + 4.5**2"

print(f"Query: {complex_math}\n")

print("Version 3 (parallel binary operations):")
result_v3_complex, turns_v3_complex = run_agent(
    complex_math,
    SYSTEM_PROMPT_V3,
    [calculate_portfolio_value_spec_v2, calculate_spec_v1],
    tool_functions_v2
)
print(f"  Result: {result_v3_complex}, Turns: {turns_v3_complex}\n")

print("Version 4 (expression evaluator):")
result_v4_complex, turns_v4_complex = run_agent(
    complex_math,
    SYSTEM_PROMPT_V4,
    [calculate_portfolio_value_spec_v2, evaluate_math_expression_spec_v4],
    tool_functions_v4
)
print(f"  Result: {result_v4_complex}, Turns: {turns_v4_complex}")

if turns_v4_complex < turns_v3_complex:
    improvement = turns_v3_complex - turns_v4_complex
    print(f"\n✅ Major improvement: Reduced from {turns_v3_complex} to {turns_v4_complex} turns ({improvement} fewer turns)!")
else:
    print(f"\n✓ Both versions completed in {turns_v4_complex} turns")

In [ ]:
test_queries = [
    "What's Tesla's stock price?",
    "How much would it cost to buy 15 shares of tesla, 24 shares of google, and 120 shares of amazon?",
    "Help me solve: 173 * 3232 + 342 / 72.1",
]

print("="*80)
print("OPTIMIZATION COMPARISON")
print("="*80)

for i, query in enumerate(test_queries, 1):
    print(f"\n{i}. Query: {query}")
    print("-" * 80)
    
    # V1: Baseline
    print("  V1 (Baseline - separate tools):")
    _, turns_v1 = run_agent(
        query,
        SYSTEM_PROMPT_V1,
        [get_stock_price_spec_v1, calculate_spec_v1],
        tool_functions_v1,
        verbose=False
    )
    print(f"    Turns: {turns_v1}")
    
    # V2: Combined portfolio tool
    print("  V2 (Combined portfolio tool):")
    _, turns_v2 = run_agent(
        query,
        SYSTEM_PROMPT_V2,
        [calculate_portfolio_value_spec_v2, calculate_spec_v1],
        tool_functions_v2,
        verbose=False
    )
    print(f"    Turns: {turns_v2}")
    
    # V3: Optimized prompt
    print("  V3 (Optimized prompt for parallel):")
    _, turns_v3 = run_agent(
        query,
        SYSTEM_PROMPT_V3,
        [calculate_portfolio_value_spec_v2, calculate_spec_v1],
        tool_functions_v2,
        verbose=False
    )
    print(f"    Turns: {turns_v3}")
    
    # Show improvement
    improvement = turns_v1 - turns_v3
    if improvement > 0:
        print(f"\n  ✅ Total improvement: {improvement} fewer turn(s) ({turns_v1} → {turns_v3})")
    elif improvement == 0:
        print(f"\n  ✓ No change needed (already optimal at {turns_v1} turn(s))")
    else:
        print(f"\n  ⚠️  Unexpected result")

print("\n" + "="*80)

## Key takeaways

### 1. Combined tools reduce turns dramatically
- Replacing `get_stock_price()` with `calculate_portfolio_value()` that handles multiple stocks at once
- Portfolio queries: **9 turns → 2 turns (78% reduction)**
- **Tradeoff**: More complex tool implementation, but worth it for common use cases

### 2. System prompt optimization encourages parallelism
- Explicitly encouraging parallel tool calls can reduce turns for complex operations
- Works when operations are truly independent (e.g., multiplication + division before addition)
- Complex math: **4 turns → 3 turns (25% reduction)**
- **Tradeoff**: Longer system prompt, minimal cost for potential savings

### 3. Expression evaluators eliminate multi-step math
- Safe AST-based expression evaluation handles entire expressions in one call
- Complex math: **3 turns → 2 turns (33% reduction)**
- Very complex expressions see even bigger gains (5+ operations in 1 call)
- **Tradeoff**: More complex tool code (~50 lines), security considerations, but major turn savings

### 4. Not all queries benefit equally
- Simple queries (single stock price) already optimal at 2 turns: call tool → return result
- Complex queries with dependencies can't always be parallelized
- Focus optimization on your most common use cases

### 5. Measuring impact is critical
- Always test turn counts before/after optimizations
- Some "optimizations" may not help if Claude already batches well
- Track metrics to justify engineering effort
- Use the `run_agent()` pattern in this notebook to measure your own optimizations

## When to use each version

- **V1 (Baseline)**: Learning/understanding only - inefficient for production
- **V2 (Combined tools)**: Good default - handles most use cases efficiently
- **V3 (Parallel hints)**: Add when you have complex multi-step operations
- **V4 (Expression evaluator)**: Best for math-heavy applications where turn count is critical

## What's next?

Further optimizations to consider:
1. **Caching**: Cache tool results for repeated queries
2. **Streaming**: Show incremental results as tools execute
3. **Prompt caching**: Reuse system prompt across requests (see Anthropic docs)
4. **Batch processing**: Process multiple independent queries in parallel

## Additional resources

- [Anthropic Tool Use Documentation](https://docs.claude.com/en/docs/agents-and-tools/tool-use)
- [Writing Tools for Agents](https://www.anthropic.com/engineering/writing-tools-for-agents)
- [Reducing Latency](https://docs.claude.com/en/docs/build-with-claude/reduce-latency)